In [10]:
import os, math, glob
import numpy as np, cv2
import torch
import torch.nn.functional as F
from skimage.measure import marching_cubes
import plotly.graph_objects as go

# --- 1. SETTINGS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
VOL_SZ = 256
UNET_DIR = "/mnt/Data1/Peoples/faiz836b/DNP-3DDMR-IR/data/organized_by_patient_unet"
PATIENT_ID = "Patient_13"

patient_path = os.path.join(UNET_DIR, PATIENT_ID)

if not os.path.exists(patient_path):
    print(f"Error: Could not find {patient_path}")
else:
    # Handle benign/malignant subfolder
    subdirs = [d for d in os.listdir(patient_path) if os.path.isdir(os.path.join(patient_path, d))]
    cat_path = os.path.join(patient_path, subdirs[0])
    
    # The 5 standard clinical views
    views = ['Right Lateral', 'Right Oblique', 'Frontal', 'Left Oblique', 'Left Lateral']
    angles_deg = [-90, -45, 0, 45, 90]
    unet_masks = []
    
    # --- 2. LOAD 2D MASKS ---
    print(f"Loading masks for {PATIENT_ID}...")
    for v in views:
        # Accommodate 'Anterior' or 'Frontal' naming conventions
        files = [f for f in glob.glob(os.path.join(cat_path, '*.png')) 
                 if v.lower() in f.lower() or (v == 'Frontal' and 'anterior' in f.lower())]
        
        if files:
            m = cv2.imread(files[0], cv2.IMREAD_GRAYSCALE)
            m = cv2.resize(m, (VOL_SZ, VOL_SZ), interpolation=cv2.INTER_NEAREST)
            m = (m > 127).astype(np.float32)
            unet_masks.append(torch.tensor(m, device=DEVICE))
        else:
            print(f"Missing view: {v}")
            
    # --- 3. SPACE CARVING (VISUAL HULL) ---
    print("Running Space Carving Intersection...")
    # Start with a solid block of clay
    hull = torch.ones((1, 1, VOL_SZ, VOL_SZ, VOL_SZ), device=DEVICE)
    
    for i, angle in enumerate(angles_deg):
        m2d = unet_masks[i].view(1, 1, VOL_SZ, VOL_SZ)
        
        # Extrude the 2D mask into a 3D block along depth (Z-axis)
        m3d = m2d.unsqueeze(2).expand(1, 1, VOL_SZ, VOL_SZ, VOL_SZ).float()
        
        # Rotate the extruded block to match the camera angle
        rad = -angle * math.pi / 180.0
        c, s = math.cos(rad), math.sin(rad)
        mat = torch.tensor([[[ c, 0, s, 0],
                             [ 0, 1, 0, 0],
                             [-s, 0, c, 0]]], dtype=torch.float32, device=DEVICE)
        
        grid = F.affine_grid(mat, hull.shape, align_corners=False)
        m3d_rot = F.grid_sample(m3d, grid, mode='bilinear', padding_mode='zeros', align_corners=False)
        
        # Chisel away the clay: Keep only what overlaps in all views
        hull = hull * m3d_rot
        
    hull_np = hull.squeeze().cpu().numpy()
    
    # --- 4. 3D VISUALIZATION ---
    print("Rendering 3D Mesh...")
    verts, faces, normals, values = marching_cubes(hull_np, level=0.5)
    
    # Swap axes for Plotly visualization (Z is depth, Y is height, X is width)
    verts = verts[:, [2, 0, 1]] 
    
    fig = go.Figure(data=[go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color='salmon', opacity=0.8,
        lighting=dict(ambient=0.4, diffuse=0.8, specular=0.2, roughness=0.5)
    )])
    
    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
            aspectmode='data',
            camera=dict(up=dict(x=0, y=1, z=0), center=dict(x=0, y=0, z=0), eye=dict(x=0, y=0, z=2))
        ),
        title=f"Classical Space Carving (Visual Hull) - {PATIENT_ID}",
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()


Loading masks for Patient_13...
Running Space Carving Intersection...
Rendering 3D Mesh...


In [13]:
import numpy as np, cv2, tifffile, glob, os
import plotly.graph_objects as go
from scipy.ndimage import gaussian_filter

# --- 1. LOAD RAW THERMAL IMAGE ---
TIFF_DIR = "/mnt/Data1/Peoples/faiz836b/DNP-3DDMR-IR/data/organized_by_patient"
PATIENT_ID = "Patient_143"
patient_path = os.path.join(TIFF_DIR, PATIENT_ID)
subdirs = [d for d in os.listdir(patient_path) if os.path.isdir(os.path.join(patient_path, d))]
cat_path = os.path.join(patient_path, subdirs[0])

files = [f for f in glob.glob(os.path.join(cat_path, '*.tiff')) if 'frontal' in f.lower() or 'anterior' in f.lower()]
raw = tifffile.imread(files[0]).astype(np.float32)
raw = cv2.resize(raw, (128, 128))

# Mask out background
raw_u8 = ((raw - raw.min()) / (raw.max() - raw.min()) * 255).astype(np.uint8)
_, mask = cv2.threshold(raw_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
mask = (mask / 255.0)

# --- 2. PSEUDO SHAPE-FROM-SHADING (Temperature = Height) ---
# Smooth it slightly so the mesh isn't completely noisy
Z = gaussian_filter(raw * mask, sigma=2)

# --- 3. 3D VISUALIZATION ---
x, y = np.meshgrid(np.arange(128), np.arange(128))
fig = go.Figure(data=[go.Surface(z=Z, x=x, y=y, colorscale='Inferno')])
fig.update_layout(
    title='Thermal Shape-from-Shading (Temperature = Height)', 
    scene=dict(aspectmode='data')
)
fig.show()


In [17]:
import numpy as np, cv2, tifffile, glob, os, math
import plotly.graph_objects as go
from scipy.ndimage import gaussian_filter

# --- 1. SETTINGS ---
TIFF_DIR = "/mnt/Data1/Peoples/faiz836b/DNP-3DDMR-IR/data/organized_by_patient"
PATIENT_ID = "Patient_143"
patient_path = os.path.join(TIFF_DIR, PATIENT_ID)
subdirs = [d for d in os.listdir(patient_path) if os.path.isdir(os.path.join(patient_path, d))]
cat_path = os.path.join(patient_path, subdirs[0])

views = ['Right Lateral', 'Right Oblique', 'Frontal', 'Left Oblique', 'Left Lateral']
angles_deg = [-90, -45, 0, 45, 90]
colors = ['red', 'orange', 'yellow', 'green', 'blue'] # Color-code each camera

fig = go.Figure()

print("Fusing 5 Thermal Topographies...")
for i, view in enumerate(views):
    # Load Image
    files = [f for f in glob.glob(os.path.join(cat_path, '*.tiff')) 
             if view.lower() in f.lower() or (view == 'Frontal' and 'anterior' in f.lower())]
    if not files:
        continue
        
    raw = tifffile.imread(files[0]).astype(np.float32)
    raw = cv2.resize(raw, (128, 128))
    
    # Mask
    raw_u8 = ((raw - raw.min()) / (raw.max() - raw.min()) * 255).astype(np.uint8)
    _, mask = cv2.threshold(raw_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = (mask / 255.0)
    
    # Intensity (Z-depth)
    intensity = (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)
    # The hotter it is, the further it pushes out
    depth = gaussian_filter(intensity * mask, sigma=2) * 35.0 
    
    # Create Local Coordinates (X, Y)
    x_local, y_local = np.meshgrid(np.linspace(-64, 64, 128), np.linspace(-64, 64, 128))
    
    # Filter to keep only the breast region (Drop background)
    valid = mask > 0.5
    x_val = x_local[valid]
    y_val = y_local[valid]
    z_val = depth[valid] + 20.0 # Add base radius to push it outward from center
    
    # Rotate to Global Coordinates based on Camera Angle
    rad = angles_deg[i] * math.pi / 180.0
    c, s = math.cos(rad), math.sin(rad)
    
    X_global = x_val * c + z_val * s
    Y_global = y_val
    Z_global = -x_val * s + z_val * c
    
    # Add to Plotly Point Cloud
    fig.add_trace(go.Scatter3d(
        x=X_global, y=Y_global, z=Z_global,
        mode='markers',
        marker=dict(size=1.5, color=colors[i], opacity=0.8),
        name=view
    ))

fig.update_layout(
    title='5-View Thermal Topography (Fused Point Cloud)', 
    scene=dict(aspectmode='data', 
               camera=dict(up=dict(x=0, y=1, z=0), center=dict(x=0, y=0, z=0), eye=dict(x=0, y=0, z=-2))),
    margin=dict(l=0, r=0, b=0, t=40)
)
fig.show()


Fusing 5 Thermal Topographies...


In [18]:
!uv pip install open3d

Resolved 73 packages in 2.15s                                        
⠙ Preparing packages... (0/13)                                                  
⠙ Preparing packages... (0/13)------------------     0 B/8.26 KiB            
⠙ Preparing packages... (0/13)------------------     0 B/8.26 KiB            
blinker              ------------------------------     0 B/8.26 KiB
⠙ Preparing packages... (0/13)------------------     0 B/27.14 KiB           
blinker              ------------------------------ 8.26 KiB/8.26 KiB
⠙ Preparing packages... (0/13)------------------     0 B/27.14 KiB           
blinker              ------------------------------ 8.26 KiB/8.26 KiB
⠙ Preparing packages... (0/13)-------------- 14.80 KiB/27.14 KiB         
blinker              ------------------------------ 8.26 KiB/8.26 KiB
⠙ Preparing packages... (0/13)-------------- 14.80 KiB/27.14 KiB         
blinker              ------------------------------ 8.26 KiB/8.26 KiB
⠙ Preparing packages... (0/13)----------

In [19]:
import numpy as np, cv2, tifffile, glob, os, math
import plotly.graph_objects as go
from scipy.ndimage import gaussian_filter
import open3d as o3d

# --- 1. SETTINGS ---
TIFF_DIR = "/mnt/Data1/Peoples/faiz836b/DNP-3DDMR-IR/data/organized_by_patient"
PATIENT_ID = "Patient_143"

patient_path = os.path.join(TIFF_DIR, PATIENT_ID)
subdirs = [d for d in os.listdir(patient_path) if os.path.isdir(os.path.join(patient_path, d))]
cat_path = os.path.join(patient_path, subdirs[0])

views = ['Right Lateral', 'Right Oblique', 'Frontal', 'Left Oblique', 'Left Lateral']
angles_deg = [-90, -45, 0, 45, 90]

all_points = []
all_normals = []

print(f"Loading and Inpainting 5 Views for {PATIENT_ID}...")
for i, view in enumerate(views):
    files = [f for f in glob.glob(os.path.join(cat_path, '*.tiff')) 
             if view.lower() in f.lower() or (view == 'Frontal' and 'anterior' in f.lower())]
    if not files: continue
        
    raw = tifffile.imread(files[0]).astype(np.float32)
    raw = cv2.resize(raw, (128, 128))
    
    # 1. THERMAL INPAINTING (Erase the craters!)
    raw_u8 = ((raw - raw.min()) / (raw.max() - raw.min()) * 255).astype(np.uint8)
    # Morphological closing fills in the dark, cold holes (nipples/veins)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    closed = cv2.morphologyEx(raw_u8, cv2.MORPH_CLOSE, kernel)
    
    # Mask extraction
    _, mask = cv2.threshold(closed, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = (mask / 255.0)
    
    # Smooth Intensity for SFS Extrusion
    intensity = closed.astype(np.float32) / 255.0
    depth = gaussian_filter(intensity * mask, sigma=3) * 35.0
    
    # Create Local Coordinates
    x_local, y_local = np.meshgrid(np.linspace(-64, 64, 128), np.linspace(-64, 64, 128))
    valid = mask > 0.5
    x_val = x_local[valid]
    y_val = y_local[valid]
    z_val = depth[valid] + 20.0
    
    # Rotate to Global Coordinates
    rad = angles_deg[i] * math.pi / 180.0
    c, s = math.cos(rad), math.sin(rad)
    
    X = x_val * c + z_val * s
    Y = y_val
    Z = -x_val * s + z_val * c
    
    pts = np.column_stack((X, Y, Z))
    
    # Mathematical Hack: Assume all normals point directly away from the center (0,0,0)
    # This gives Poisson a perfect set of outward-facing vectors to wrap around.
    norms = pts / np.linalg.norm(pts, axis=1, keepdims=True)
    
    all_points.append(pts)
    all_normals.append(norms)

# --- 2. POISSON SURFACE RECONSTRUCTION ---
print("Fusing Point Clouds...")
pts_merged = np.vstack(all_points)
norms_merged = np.vstack(all_normals)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pts_merged)
pcd.normals = o3d.utility.Vector3dVector(norms_merged)

print("Solving Poisson PDE to wrap surface...")
# Depth determines the resolution of the mesh
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=7)

# Clean up artifact bubbles created by Poisson in empty space
print("Cleaning mesh...")
vertices_to_remove = densities < np.quantile(densities, 0.05)
mesh.remove_vertices_by_mask(vertices_to_remove)

# --- 3. PLOTLY VISUALIZATION ---
print("Rendering Final Solid Mesh...")
verts = np.asarray(mesh.vertices)
# Swap axes for Plotly (Z is depth)
verts = verts[:, [0, 2, 1]] 
faces = np.asarray(mesh.triangles)

fig = go.Figure(data=[go.Mesh3d(
    x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    color='lightpink', opacity=1.0,
    lighting=dict(ambient=0.4, diffuse=0.8, specular=0.2, roughness=0.5)
)])

fig.update_layout(
    title='Poisson Surface Reconstruction (The Ultimate Classical Baseline)', 
    scene=dict(
        aspectmode='data',
        xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
        camera=dict(up=dict(x=0, y=1, z=0), center=dict(x=0, y=0, z=0), eye=dict(x=0, y=0, z=-2))
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)
fig.show()


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading and Inpainting 5 Views for Patient_143...
Fusing Point Clouds...
Solving Poisson PDE to wrap surface...
Cleaning mesh...
Rendering Final Solid Mesh...


In [8]:
import numpy as np, cv2, glob, os
import plotly.graph_objects as go

# --- 1. LOAD FRONTAL MASK ---
VOL_SZ = 256
UNET_DIR = "/mnt/Data1/Peoples/faiz836b/DNP-3DDMR-IR/data/organized_by_patient_unet"
PATIENT_ID = "Patient_143"
patient_path = os.path.join(UNET_DIR, PATIENT_ID)
subdirs = [d for d in os.listdir(patient_path) if os.path.isdir(os.path.join(patient_path, d))]
cat_path = os.path.join(patient_path, subdirs[0])

files = [f for f in glob.glob(os.path.join(cat_path, '*.png')) if 'frontal' in f.lower() or 'anterior' in f.lower()]
m = cv2.imread(files[0], cv2.IMREAD_GRAYSCALE)
m = cv2.resize(m, (VOL_SZ, VOL_SZ), interpolation=cv2.INTER_NEAREST)

print("Running Costa et al. 4-Step IMF Extraction...")

# --- 2. THE 4-STEP ALGORITHM ---
# Step (1): Bottom-up search (Find lowest pixel in each column)
S1 = set()
for x in range(VOL_SZ):
    y_idx = np.where(m[:, x] > 127)[0]
    if len(y_idx) > 0:
        S1.add((x, int(np.max(y_idx)))) # max y is visually the bottom of the image

# Step (2): Lateral scan from right (Find rightmost pixel in each row)
S2 = set()
for y in range(VOL_SZ):
    x_idx = np.where(m[y, :] > 127)[0]
    if len(x_idx) > 0:
        S2.add((int(np.max(x_idx)), y)) 

# Step (3): Lateral scan from left (Find leftmost pixel in each row)
S3 = set()
for y in range(VOL_SZ):
    x_idx = np.where(m[y, :] > 127)[0]
    if len(x_idx) > 0:
        S3.add((int(np.min(x_idx)), y))

# Step (4): Remove S2 and S3 points from S1
# "all pixels from step 1 that are found in any of the other 3 detections are removed"
imf_points = [p for p in S1 if p not in S2 and p not in S3]

# --- 3. EXTRACT P1, P2, P3 ---
imf_x = np.array([p[0] for p in imf_points])
# Invert Y for plotting so it looks upright
imf_y = np.array([VOL_SZ - p[1] for p in imf_points]) 

# Sort points horizontally
sort_idx = np.argsort(imf_x)
imf_x, imf_y = imf_x[sort_idx], imf_y[sort_idx]

# "P1 and P3 correspond to the inflections on the right and left" (The ends of the IMF curve)
P3_idx = 0                  # Leftmost point
P1_idx = len(imf_x) - 1     # Rightmost point

# "P2 is found at the junction of the curves in the center of the body"
# Since the breast valley dips down in image coordinates (which means Y is lowest in our inverted plot)
# P2 is the local minimum in the center!
p2_idx = np.argmin(imf_y)

print(f"P3 (Left Inflection): X={imf_x[P3_idx]}, Y={imf_y[P3_idx]}")
print(f"P2 (Center Junction): X={imf_x[p2_idx]}, Y={imf_y[p2_idx]}")
print(f"P1 (Right Inflection): X={imf_x[P1_idx]}, Y={imf_y[P1_idx]}")

# --- 4. VISUALIZE RESULT ---
fig = go.Figure()

# Plot the raw mask outline lightly
contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
c = max(contours, key=cv2.contourArea).squeeze()
fig.add_trace(go.Scatter(x=c[:, 0], y=VOL_SZ-c[:, 1], mode='lines', line=dict(color='gray', width=1, dash='dot'), name='U-Net Boundary'))

# Plot the extracted IMF
fig.add_trace(go.Scatter(x=imf_x, y=imf_y, mode='lines', line=dict(color='yellow', width=4), name='Extracted IMF (Step 4)'))

# Plot the Anchors
fig.add_trace(go.Scatter(x=[imf_x[P3_idx]], y=[imf_y[P3_idx]], mode='markers', marker=dict(color='blue', size=12), name='P3'))
fig.add_trace(go.Scatter(x=[imf_x[p2_idx]], y=[imf_y[p2_idx]], mode='markers', marker=dict(color='red', size=12), name='P2'))
fig.add_trace(go.Scatter(x=[imf_x[P1_idx]], y=[imf_y[P1_idx]], mode='markers', marker=dict(color='green', size=12), name='P1'))

fig.update_layout(title="Costa et al. IMF Extraction (Figure 4 & 5)", height=500, xaxis=dict(scaleanchor="y", scaleratio=1))
fig.show()


Running Costa et al. 4-Step IMF Extraction...
P3 (Left Inflection): X=41, Y=63
P2 (Center Junction): X=63, Y=31
P1 (Right Inflection): X=208, Y=55
